# 假设检验的第一类错误率

> 对应代码：`MathStatsCode/code_in_notes/Chap.12/test_under_null.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebook/PyStata.md`。

**实验目的**：在 H0（μ=μ0）下重复10000次 t 检验，验证5%显著性水平下的实际拒绝率约等于5%。


首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.12`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。


In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.12")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

`clear`、`set more off`、`set seed 880505`：清空数据、关闭分页、固定随机种子，为重复模拟做准备。


In [ ]:
%%stata
clear
set more off
set seed 880505


**test under H0: mu=mu0 at 5% significant level**

本段验证**假设检验的第一类错误率**（检验水平）：
- 定义程序 `test_normal_mean`：`syntax` 接受样本量、真实均值 mu、原假设均值 mu0、标准差等参数；生成 n 个 N(μ,σ²) 样本后计算 t 统计量 `√n(x̄-μ0)/s`；用 `if abs(t) > invt(n-1, 1-0.025)` 判断是否落入拒绝域（双侧、5% 水平，临界值 t_{0.975}(9)=2.262）；用标量 `rejected` 记录结果并通过 `return scalar rejected=` 返回；
- `simulate rejected=r(rejected), reps(10000)` 把整个检验重复10000次；
- `su rejected` 给出**拒绝频率**。因为默认参数下 mu=mu0=0（原假设为真），拒绝频率就是实际的第一类错误率——理论上应为5%，模拟结果应接近该值，说明 t 检验在名义水平上是“准”的。


In [ ]:
%%stata
cap program drop test_normal_mean
program define test_normal_mean, rclass
	version 12
	syntax [,obs(integer 10) mu(real 0) mu0(real 0) sigma(real 1)]
	drop _all
	set obs `obs'
	tempvar x
	gen `x'=`sigma'*rnormal()+`mu'
	quietly: su `x'
	tempname std_mean rejected
	scalar `std_mean'=sqrt(`obs')*(r(mean)-`mu0')/r(sd)
	di `std_mean'
	if abs(`std_mean')>invt(`obs'-1,1-0.05/2){
		scalar `rejected'=1
	}
	else{
		scalar `rejected'=0
	}
	return scalar rejected=`rejected'
end

simulate rejected=r(rejected),reps(10000): test_normal_mean
su rejected
